# Library Implementation

In [ ]:
import numpy as np
from sklearn import datasets
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

#(MNIST-like digits)
digits = datasets.load_digits()
X = digits.data
y = digits.target

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42
)

# Step (b): Select 100 labeled samples
n_labeled = 100
indices = np.random.permutation(len(X_train))

labeled_idx = indices[:n_labeled]
unlabeled_idx = indices[n_labeled:]

# Step (c): Create semi-supervised labels
y_semi = np.copy(y_train)
y_semi[unlabeled_idx] = -1   # unlabeled = -1

X_labeled = X_train[labeled_idx]
y_labeled = y_train[labeled_idx]

X_unlabeled = X_train[unlabeled_idx]
y_unlabeled_true = y_train[unlabeled_idx]  # ground truth for evaluation

In [2]:
from sklearn.semi_supervised import SelfTrainingClassifier
from sklearn.svm import SVC

base_model = SVC(probability=True, gamma='auto')

self_model = SelfTrainingClassifier(base_model)
self_model.fit(X_train, y_semi)

# Predict pseudo-labels for unlabeled data
pseudo_labels = self_model.predict(X_unlabeled)

acc = accuracy_score(y_unlabeled_true, pseudo_labels)
print("Self-Training Accuracy:", acc)

Self-Training Accuracy: 0.10458081244598098


In [ ]:
import numpy as np
from sklearn import datasets
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.naive_bayes import GaussianNB
from mvlearn.semi_supervised import CTClassifier

digits = datasets.load_digits()
X = digits.data
y = digits.target

# Binary filter (IMPORTANT)
mask = (y == 0) | (y == 1)
X = X[mask]
y = y[mask]

# Split into 2 views
X1 = X[:, :32]
X2 = X[:, 32:]

# Train-test split
X1_train, X1_test, X2_train, X2_test, y_train, y_test = train_test_split(
    X1, X2, y, test_size=0.3, random_state=42
)

# Create unlabeled indices
np.random.seed(42)
n_unlabeled = int(0.5 * len(y_train))
unlabeled_idx = np.random.choice(len(y_train), n_unlabeled, replace=False)

y_semi = y_train.astype(float)
y_semi[unlabeled_idx] = np.nan  

# Models
clf1 = GaussianNB()
clf2 = GaussianNB()

# Co-Training
ct = CTClassifier(clf1, clf2)
ct.fit([X1_train, X2_train], y_semi)

# Predict pseudo-labels
pseudo_labels = ct.predict([
    X1_train[unlabeled_idx],
    X2_train[unlabeled_idx]
])

# Accuracy
acc = accuracy_score(y_train[unlabeled_idx], pseudo_labels)
print("Co-Training Accuracy:", acc)

Co-Training Accuracy: 0.9920634920634921


In [4]:
from sklearn.cluster import KMeans

kmeans = KMeans(n_clusters=10, random_state=42)
clusters = kmeans.fit_predict(X_train)

cluster_labels = {}

for i in range(10):
    points = np.where(clusters == i)[0]
    labels = y_train[points]

    if len(labels) > 0:
        cluster_labels[i] = np.bincount(labels).argmax()

# Predict unlabeled
unlabeled_clusters = kmeans.predict(X_unlabeled)
pseudo_labels = np.array([cluster_labels[c] for c in unlabeled_clusters])

acc = accuracy_score(y_unlabeled_true, pseudo_labels)
print("Clustering Accuracy:", acc)

/home/soni/mvlearn_env/lib/python3.9/site-packages/sklearn/cluster/_kmeans.py:870: FutureWarning: The default value of `n_init` will change from 10 to 'auto' in 1.4. Set the value of `n_init` explicitly to suppress the warning
  warnings.warn(


Clustering Accuracy: 0.7951598962834918


# Manual Implementation

In [5]:
def knn(X_train, y_train, X_test, k=3):
    preds = []
    for x in X_test:
        dist = np.linalg.norm(X_train - x, axis=1)
        idx = dist.argsort()[:k]
        labels = y_train[idx]
        preds.append(np.bincount(labels).argmax())
    return np.array(preds)

# Train on labeled
pseudo_labels = knn(X_labeled, y_labeled, X_unlabeled)

acc = accuracy_score(y_unlabeled_true, pseudo_labels)
print("Manual Self-Training Accuracy:", acc)

Manual Self-Training Accuracy: 0.8522039757994814


In [6]:
# Split into two views
X1 = X_train[:, :32]
X2 = X_train[:, 32:]

X1_l = X1[labeled_idx]
X2_l = X2[labeled_idx]

y_l = y_labeled

X1_u = X1[unlabeled_idx]
X2_u = X2[unlabeled_idx]

# Each model predicts
pred1 = knn(X1_l, y_l, X1_u)
pred2 = knn(X2_l, y_l, X2_u)

# Combine predictions (agreement)
final_pred = []

for i in range(len(pred1)):
    if pred1[i] == pred2[i]:
        final_pred.append(pred1[i])
    else:
        final_pred.append(pred1[i])  # simple choice

final_pred = np.array(final_pred)

acc = accuracy_score(y_unlabeled_true, final_pred)
print("Manual Co-Training Accuracy:", acc)

Manual Co-Training Accuracy: 0.6577355229040622


In [7]:
def kmeans(X, k=10, iters=10):
    centroids = X[np.random.choice(len(X), k, replace=False)]

    for _ in range(iters):
        dist = np.linalg.norm(X[:, None] - centroids, axis=2)
        clusters = np.argmin(dist, axis=1)

        new_centroids = []
        for i in range(k):
            pts = X[clusters == i]
            if len(pts) > 0:
                new_centroids.append(pts.mean(axis=0))
            else:
                new_centroids.append(centroids[i])
        centroids = np.array(new_centroids)

    return clusters, centroids


clusters, centroids = kmeans(X_train)

cluster_labels = {}

for i in range(10):
    labels = y_train[clusters == i]
    if len(labels) > 0:
        cluster_labels[i] = np.bincount(labels).argmax()

# Predict
dist = np.linalg.norm(X_unlabeled[:, None] - centroids, axis=2)
unlabeled_clusters = np.argmin(dist, axis=1)

pseudo_labels = np.array([cluster_labels[c] for c in unlabeled_clusters])

acc = accuracy_score(y_unlabeled_true, pseudo_labels)
print("Manual KMeans Accuracy:", acc)

Manual KMeans Accuracy: 0.7346585998271391
